# PREPROCESSING 

The data that we need is:

1. CERF allocations 
2. IDMC for displacements 
3. ACLED for fatalities 
4. HDX Signals 
5. EconAI 
6. INFORM Index
7. Severity Index

In this nootbook we will upload the data and preprocess it in order to have it in an apropiate format to start feature enginyeering. So for each dataset we have to clean the data, filter it to keep only the relevant information and produce the final files that we will use during the project. All the files should have `iso3` and `month` as columns, because these are going to be the MultiIndex of our final dataset.

In [3]:
import pandas as pd
import os
import pycountry
import numpy as np
from pathlib import Path

### CERF ALLOCATIONS

Regarding the data of the CERF allocations, we have 2 files, one with data from 2006 to 2024, and one with data of 2024 and 2025. One issue is that the formats of the data are different, so we need to create a file with data from 2006 to 2025 unifying the two formats and keeping only the relevant information. 

In [29]:
cerf_0624 = pd.read_excel("../data_raw/CERF allocations/CERF allocations 2006-Jun2024 - EconAI.xlsx", skiprows=1) # Because there's one line of text that's no need it
cerf_2425 = pd.read_excel("../data_raw/CERF allocations/CERF allocations 2024-2025.xlsx")

In [30]:
cerf_0624.head()

,Application Code,Country,Emergency Type,Application Title,Year,Amount Approved,Total Amount Required,Date of ERC Endorsement,Amount Endorsed by ERC,Geographical Areas of Implementation,...,Number of Children,Adults,Number of IDPs,Number of Returnees,Number of Refugees,Number of Host Population,Number of Other Affected People,Persons with Disabilities,2a. Overview of the humanitarian situation,2b. CERF-funded assistance
0,24-RR-BDI-65155,Burundi,Flood,Burundi RR Application May 2024 (El Nino-relat...,2024,2500773,26000000,2024-05-18,2500000,"Communes de Mutimbuzi, Kabezi, Mubimbi (Provin...",...,28142.0,31858.0,35315.0,8180.0,4478.0,12027.0,NaN,4169.0,Heavy rains induced by El Niño have caused sev...,This $2.5 million CERF allocation aims to prov...
1,24-RR-BFA-65175,Burkina Faso,Violence/Clashes,Burkina Faso RR Application May 2024 (Violence...,2024,5000007,934600000,2024-05-18,5000000,"Soum et Yagha (Sahel), Bam, Sanmatenga et Name...",...,67019.0,50681.0,NaN,81690.0,1000.0,35010.0,NaN,7002.0,Burkina Faso is facing increasing humanitarian...,"In response to the crisis, the Emergency Relie..."
2,24-RR-ZWE-64774,Zimbabwe,Drought,Zimbabwe RR Application May 2024 (El Niño-rela...,2024,3000727,429300000,2024-04-29,3000000,"Beitbridge, Binga, Bikita, Buhera, Bulilima, M...",...,85000.0,3600.0,NaN,NaN,NaN,88600.0,NaN,30.0,The El Niño-induced drought has severely exace...,This additional $3 million allocation aims to ...
3,24-RR-MWI-64768,Malawi,Drought,Malawi RR Application May 2024 (El Nino - Drou...,2024,1995676,445000000,2024-04-29,2000000,"Machinga, Nsanje districts",...,167126.0,69119.0,NaN,NaN,NaN,55000.0,181245.0,5282.0,"Prolonged dry spells, many of which lasting lo...","In response, the Emergency Relief Coordinator ..."
4,24-RR-NPL-65080,Nepal,Flood,Nepal RR Application May 2024 (Anticipatory Ac...,2024,2724993,2664091,NaT,2664091,"Sunsari, Saptari, Bardiya and Kailali",...,98479.0,185352.0,NaN,NaN,NaN,NaN,NaN,5359.0,The flat plains of the Terai in Nepal are pron...,BACKGROUND: The Emergency Relief Coordinator s...


In [31]:
len(cerf_0624)

1172

In [32]:
cerf_2425.head()

,Allocation Code,Allocation Type,Allocation Status,Allocation Source Name,Is AA Allocation,CERF Website Published Year,Continent Name,Region Name,Country Name,Emergency Types,Emergency Group for Global Reporting,Allocation Year,Total Budget Requested,Amount Approved,Total Amount Required For Response,Total Amount Received For Response,Total People Affected By Crisis,ERCEndorsementDate
0,CERF-AGO-25-RR-1468,CERF Rapid Response: Angola May 2025 (Cholera),Under Final Reporting,Rapid Response,No,2025.0,Africa,Middle Africa,Angola,Disease Outbreak - Cholera,Disease outbreak,2025.0,1800000.0,1799873.54,17000000.0,4101414.0,6045650.0,2025-05-09T00:00:00
1,CERF-BDI-24-UF-1406,CERF Underfunded Emergencies: Burundi 2024 (Po...,Under Implementation,Underfunded Emergencies,No,2024.0,Africa,Eastern Africa,Burundi,Climate / natural disaster - Flood,Climate / natural disaster,2024.0,6000000.0,5992935.38,26000000.0,12700000.0,306000.0,2024-08-15T00:00:00
2,CERF-BDI-25-RR-1461,CERF Rapid Response: Burundi Mar 2025 (Displac...,Under Final Reporting,Rapid Response,No,2025.0,Africa,Eastern Africa,Burundi,Conflict - Displacement,Conflict,2025.0,2500000.0,2499131.75,62200000.0,8000000.0,70000.0,2025-03-27T00:00:00
3,CERF-BDI-25-RR-1511,CERF Rapid Response: Burundi Dec 2025 (Refugees),Under Implementation,Rapid Response,No,2026.0,Africa,Eastern Africa,Burundi,Conflict - Refugees,Conflict,2025.0,3500000.0,3500167.98,35300000.0,3400000.0,80000.0,2025-12-29T00:00:00
4,CERF-BFA-24-UF-1410,CERF Underfunded Emergencies: Burkina Faso 202...,Under Implementation,Underfunded Emergencies,No,2024.0,Africa,Western Africa,Burkina Faso,Conflict - Violence/clashes,Conflict,2024.0,11000000.0,11000015.19,934600000.0,339600000.0,1300000.0,2024-08-15T00:00:00


In [33]:
len(cerf_2425)

126

The filters that we have to apply are the following:

1. Only keep Rapid Response.
2. Filter out Anticipatory Action Allocations.
3. Only use allocations with Emergency Types = Displacement, Human Rights or Violence/Clashes.

To do so it will depend on each file, since they don't have the same columns, buit yet the relevant information is there for both.

**Keeping only RR**

The file 0624 already contains only this allocations, so we only need to do it for the 2425 file. The Rapid Response allocations are the ones which it's Allocation Type contains Rapid Response in it:

In [34]:
cerf_2425 = cerf_2425[cerf_2425['Allocation Type'].str.contains('Rapid Response', case=False, na=False)].copy()
len(cerf_2425)

96

**Filter out Anticipatory Action**

In the first file (2006-2024) we can detect if it'a an Anticipatory Action looking at the Application Title column (it contains Anticipatory Action in the text of this column). In the case of the second file (2024-2025) there's a column `AA Allocations`that contains either a Yes or a No.

In [35]:
cerf_0624 = cerf_0624[~cerf_0624['Application Title'].str.contains('Anticipatory Action', case=False, na=False)].copy()
cerf_2425 = cerf_2425[cerf_2425['Is AA Allocation'].str.strip() != 'Yes'].copy()

In [36]:
len(cerf_0624)

1139

In [37]:
len(cerf_2425)

70

**Keep only Displacement, Human Rights or Violence/Clashes**

In the first file (2006-2024) this information is contained in the column `Emergency Type` with the names "Displacement", "Human Rights" and "Violence/Clashes". In the second one (2024-2025) `Emergency Types` under the names: "Confict - Displacement", "Conflict - Displacement, Conflict - Refugees" and "Confict - Violence/clashes".

In [38]:
condition_0624 = (
    cerf_0624['Emergency Type'].str.contains('Displacement', case=False, na=False) |
    cerf_0624['Emergency Type'].str.contains('Human Rights', case=False, na=False) |
    cerf_0624['Emergency Type'].str.contains('Violence', case=False, na=False)
)
cerf_0624 = cerf_0624[condition_0624].copy()

In [39]:
condition_2425 = (
    cerf_2425['Emergency Types'].str.contains('Displacement', case=False, na=False) |
    cerf_2425['Emergency Types'].str.contains('Violence', case=False, na=False)
)
cerf_2425 = cerf_2425[condition_2425].copy()

In [40]:
len(cerf_0624)

347

In [41]:
len(cerf_2425)

20

Now that the data is filtered, we are going to create a final csv only containing the columns that we want. We want each row to be an allocation, containing the iso3 (Country code), County Name, Allocation Date (First Project Approved Date) and Amount Approved.

All this information is in the files except the iso3 code, that we will have to manually put it.

In [42]:
# We have added "Cote d'Ivoire", "Democratic Republic of the Congo", "Venezuela Regional Refugee and Migration Crisis" and "Palestinian territory, occupied", "occupied Palestinian territory".
manual_iso3 = {
    "Brunei": "BRN",
    "East Timor": "TLS",
    "Micronesia": "FSM",
    "Bailiwick of Guernsey": "GGY",
    "Bailiwick of Jersey": "JEY",
    "Kosovo": "XKX",  # common non-ISO code
    "Russia": "RUS",
    "Vatican City": "VAT",
    "Caribbean Netherlands": "BES",
    "Curacao": "CUW",
    "Falkland Islands": "FLK",
    "Saint-Barthelemy": "BLM",
    "Saint-Martin": "MAF",
    "Sint Maarten": "SXM",
    "Palestine": "PSE",
    "occupied Palestinian territory": "PSE",
    "Palestinian territory, occupied": "PSE",
    "Turkey": "TUR",
    "Cape Verde": "CPV",
    "Democratic Republic of Congo": "COD",
    "Democratic Republic of the Congo": "COD",
    "Ivory Coast": "CIV",
    "Cote d'Ivoire": "CIV",
    "Republic of Congo": "COG",
    "Reunion": "REU",
    # no official ISO 3166-1 code for this one, keep as custom if needed
    "Akrotiri and Dhekelia": "AKD",
    "Venezuela Regional Refugee and Migration Crisis": "VEN"
}

def country_to_iso3(name):
    # manual first
    if name in manual_iso3:
        return manual_iso3[name]
    
    # then try pycountry lookup
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None


In [43]:
# ---- Apply mapping ----
cerf_0624["iso3"] = cerf_0624["Country"].apply(country_to_iso3)
cerf_2425["iso3"] = cerf_2425["Country Name"].apply(country_to_iso3)

In [44]:
cerf_0624[cerf_0624["iso3"].isna()]

,Application Code,Country,Emergency Type,Application Title,Year,Amount Approved,Total Amount Required,Date of ERC Endorsement,Amount Endorsed by ERC,Geographical Areas of Implementation,...,Adults,Number of IDPs,Number of Returnees,Number of Refugees,Number of Host Population,Number of Other Affected People,Persons with Disabilities,2a. Overview of the humanitarian situation,2b. CERF-funded assistance,iso3
206,20-RR-GLB-46341,Global,Human Rights,Global RR Application Dec 2020 (GBV programming),2021,25004109,218134542,2020-10-29,25000000,"Bangladesh, Cameroon, Colombia, Ethiopia, Iraq...",...,482674.0,279991.0,127198.0,103530.0,176692.0,79516.0,34328.0,The COVID-19 situation exacerbated already exi...,The United Nations Population Fund (UNFPA) and...,NaN


In [45]:
cerf_2425[cerf_2425["iso3"].isna()]

,Allocation Code,Allocation Type,Allocation Status,Allocation Source Name,Is AA Allocation,CERF Website Published Year,Continent Name,Region Name,Country Name,Emergency Types,Emergency Group for Global Reporting,Allocation Year,Total Budget Requested,Amount Approved,Total Amount Required For Response,Total Amount Received For Response,Total People Affected By Crisis,ERCEndorsementDate,iso3


In the file 2006-2024, the Country Name is in the column `Country`, the Allocation Date is in `Date of Earliest Project Start`, and Amount Approved in `Amount Approved`. In the other one, the Country Name is in the column `Country Name`, the Allocation Date is in `??`, and Amount Approved in `Amount Approved`.

First we are deletting all the rows that don't have the appropiate information such as what we will consider the Allocation Date and the Country Code, and finally we will joint the two files in one.

In [46]:
cerf_0624 = cerf_0624.dropna(subset=['iso3']).copy()
cerf_2425 = cerf_2425.dropna(subset=['iso3']).copy()

In [47]:
cerf_0624[cerf_0624["Date of Earliest Project Start"].isna()]

,Application Code,Country,Emergency Type,Application Title,Year,Amount Approved,Total Amount Required,Date of ERC Endorsement,Amount Endorsed by ERC,Geographical Areas of Implementation,...,Adults,Number of IDPs,Number of Returnees,Number of Refugees,Number of Host Population,Number of Other Affected People,Persons with Disabilities,2a. Overview of the humanitarian situation,2b. CERF-funded assistance,iso3
354,18-RR-COD-28607,Democratic Republic of the Congo,Displacement,DR Congo RR Application Jan 2018 (South Sudan ...,2018,0,5753307,NaT,0,Provinces de l’Ituri et du Haut Uele,...,32431.0,NaN,NaN,88976.0,NaN,NaN,NaN,NaN,NaN,COD
477,15-RR-DZA-15966,Algeria,Displacement,15-RR-DZA-15966_Algeria_Aug2015_Application,2015,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DZA
524,14-RR-SYR-12202,Syrian Arab Republic,Displacement,14-RR-SYR-12202_Syria_Oct2014_Application,2014,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SYR
642,12-RR-MLI-13485,Mali,Displacement,12-RR-MLI-13485_Mali_Jun2016_Application,2012,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MLI
649,12-RR-BDI-13271,Burundi,Displacement,12-RR-BDI-13271_Burundi_Jun2016_Application,2012,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BDI
676,12-RR-NAM-8210,Namibia,Displacement,12-RR-NAM-8210_Namibia_Jun2016_Application,2012,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NAM
687,13-RR-LBN-7018,Lebanon,Displacement,13-RR-LBN-7018_Lebanon_Jun2016_Application,2012,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LBN
721,11-RR-LBY-13475,Libya,Displacement,11-RR-LBY-13475_Libya_Jun2016_Application,2011,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LBY
725,11-RR-LBY-13068,Libya,Displacement,11-RR-LBY-13068_Libya_Jun2016_Application,2011,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LBY
726,11-RR-LBY-13064,Libya,Displacement,11-RR-LBY-13064_Libya_Jun2016_Application,2011,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LBY


In [48]:
cerf_0624 = cerf_0624.dropna(subset=['Date of Earliest Project Start']).copy()
len(cerf_0624)

331

In [49]:
cerf_0624["Country Name"] = cerf_0624["Country"]
cerf_0624["Allocation Date"] = cerf_0624["Date of Earliest Project Start"]

In [50]:
cerf_2425[cerf_2425["ERCEndorsementDate"].isna()]

,Allocation Code,Allocation Type,Allocation Status,Allocation Source Name,Is AA Allocation,CERF Website Published Year,Continent Name,Region Name,Country Name,Emergency Types,Emergency Group for Global Reporting,Allocation Year,Total Budget Requested,Amount Approved,Total Amount Required For Response,Total Amount Received For Response,Total People Affected By Crisis,ERCEndorsementDate,iso3


In [51]:
cerf_2425 = cerf_2425.dropna(subset=['ERCEndorsementDate']).copy()
len(cerf_2425)

20

In [52]:
cerf_2425["Allocation Date"] = cerf_2425["ERCEndorsementDate"]

In [53]:
cols_to_keep = ["iso3", "Country Name", "Allocation Date", "Amount Approved"]

df1_clean = cerf_0624[cols_to_keep]
df2_clean = cerf_2425[cols_to_keep]

cerf_clean = pd.concat([df1_clean, df2_clean], ignore_index=True)

cerf_clean.to_csv("../data_clean/cerf_clean.csv", index=False)

### IDMC

The IDMC data consist of one file with the wrangled data.

In [54]:
idmc = pd.read_csv("../data_raw/IDMC/idmc_conflict_wrangled_20260424.csv", sep=",")
idmc.head()

,iso3,displacement_type,date,displacement_daily,displacement_7d,displacement_30d
0,AB9,Conflict,1/1/2018,0.0,NaN,NaN
1,AB9,Conflict,1/2/2018,0.0,NaN,NaN
2,AB9,Conflict,1/3/2018,0.0,NaN,NaN
3,AB9,Conflict,1/4/2018,0.0,NaN,NaN
4,AB9,Conflict,1/5/2018,0.0,NaN,NaN


First we filter so that we only keep the ones where the `displacement_type`= "Conflict". Then we create the variable `month` from the `date` and we group by `iso3` and `month`. We also delete all the observations that doesen't have `date` or `iso3`.

In [55]:
idmc = idmc[idmc["displacement_type"].astype(str).str.strip() == "Conflict"].copy()

# Convert date to datetime
idmc["date"] = pd.to_datetime(
    idmc["date"],
    format="%m/%d/%Y",  # e.g. 01/01/2018
    errors="coerce"
)
idmc = idmc.dropna(subset=["date"])
idmc = idmc.dropna(subset=["iso3"])

# Extract year-month
idmc["month"] = idmc["date"].dt.to_period("M").dt.to_timestamp()

# Aggregate monthly displacement (conflict only)
idmc = (
    idmc
    .groupby(["iso3", "month"])["displacement_daily"]
    .sum()
    .reset_index()
    .rename(columns={"displacement_daily": "monthly_displacement"})
)

idmc.to_csv("../data_clean/idmc_clean.csv", index=False)

In [56]:
idmc.head()

,iso3,month,monthly_displacement
0,AB9,2018-01-01,0.0
1,AB9,2018-02-01,0.0
2,AB9,2018-03-01,0.0
3,AB9,2018-04-01,0.0
4,AB9,2018-05-01,0.0


### ACLED

The data from acled is a csv file containing data of each event. This complete dataset contains the `notes` column that will be very helpful.

In [4]:
acled = pd.read_csv("../data_raw/ACLED/acled_data_raw_20260313.csv", sep=",")
acled.head()

,Unnamed: 0,event_date,event_type,iso,latitude,longitude,notes,fatalities
0,1,2019-03-06,Protests,266,0.3901,9.4544,06 March. Teachers protest by arranging a sit-...,0
1,2,2019-03-01,Protests,266,0.3901,9.4544,01 March. Transport workers from the National ...,0
2,3,2019-02-11,Strategic developments,430,6.3100,-10.8000,11 February. Roots FM station in Monrovia is a...,0
3,4,2019-02-02,Violence against civilians,404,-1.2692,36.6713,02 February: Two dead bodies were found on the...,2
4,5,2019-01-31,Strategic developments,430,6.3100,-10.8000,31 January. 3 armed men break into the Roots F...,0


First notice that the iso codes are in integer format, so we need to convert them to the iso with letters, to match all the other databases. We have a problem with 2 iso, that are 0 and 2. 0 represents Kosovo, and 2 International Waters.

In [5]:
import country_converter as coco

# 1. OPTIMIZED COUNTRY CONVERSION (With custom fixes for Kosovo & Maritime events)
# Extract unique codes but exclude 0 and 2 from the library's automatic lookup
unique_codes = acled['iso'].dropna().unique()
codes_for_library = [int(x) for x in unique_codes if int(x) not in [0, 2]]

# Translate only the standard official codes (takes milliseconds)
cc = coco.CountryConverter()
unique_iso3 = cc.convert(names=codes_for_library, to='ISO3')

if isinstance(unique_iso3, str):
    unique_iso3 = [unique_iso3]

# Create the standard translation dictionary
country_map = dict(zip(codes_for_library, unique_iso3))

# HARDCODE FIXES: Manually map the custom placeholders used by ACLED
country_map[0] = 'XKX'  # 'XKX' is the widely accepted user-defined ISO3 code for Kosovo
country_map[2] = np.nan  

# Map the dictionary to the entire dataframe instantly
acled['iso3'] = acled['iso'].map(country_map)

In [6]:
acled[acled["event_date"].isna()]

,Unnamed: 0,event_date,event_type,iso,latitude,longitude,notes,fatalities,iso3


In [7]:
acled[acled["iso3"].isna()]

,Unnamed: 0,event_date,event_type,iso,latitude,longitude,notes,fatalities,iso3
1797112,1797113,2024-09-29,Protests,2,34.6007,32.9561,"On 29 September 2024, a couple of hundred pro-...",0,NaN
1797113,1797114,2021-01-21,Protests,2,35.0950,33.9000,"On 21 January 2021, Turkish Cypriot workers li...",0,NaN
1797114,1797115,2019-02-01,Strategic developments,2,35.0950,33.9000,"On Feb. 1 2019, Turkish military forces moved ...",0,NaN
1797115,1797116,2018-06-10,Protests,2,34.6007,32.9561,"On 10 June 2018, thousands of people wearing T...",0,NaN
1797116,1797117,2024-01-14,Protests,2,34.6007,32.9561,"On 14 January 2024, pro-Palestinian activists ...",0,NaN
1797117,1797118,2023-07-10,Protests,2,35.0183,33.8098,"On 10 July 2023, quarry workers drove 50 truck...",0,NaN
1797118,1797119,2022-06-03,Protests,2,34.6654,32.8857,"On 3 June 2022, on the platinum jubilee of Que...",0,NaN
1797119,1797120,2022-02-20,Protests,2,34.6007,32.9561,"On 20 February 2022, demonstrators organized b...",0,NaN
1797120,1797121,2021-09-18,Riots,2,35.0183,33.8098,"On 18 September 2021, a group of poachers driv...",0,NaN
1797121,1797122,2021-01-25,Protests,2,35.0950,33.9000,"On 25 January 2021, Turkish Cypriot workers li...",0,NaN


In [8]:
acled = acled.dropna(subset=['iso3']).copy()

In [9]:
acled[acled["event_type"].isna()]

,Unnamed: 0,event_date,event_type,iso,latitude,longitude,notes,fatalities,iso3


In [10]:
acled["event_type"].value_counts()

event_type
Protests                      1185415
Explosions/Remote violence     533970
Battles                        489886
Violence against civilians     331382
Strategic developments         211637
Riots                          187335
Name: count, dtype: int64

To save the `event_type` column usefully, we are creating dummy variables of all the possible types:

In [11]:
dummies = pd.get_dummies(acled['event_type'], dtype=int)
acled = pd.concat([acled, dummies], axis=1)
event_types = dummies.columns.tolist()

We want the data agregatted by month x country. We are going to keep only `fatalities` and `notes`, so the number of fatalities is going to be the sum of the fatalities at that month, and the notes is going to be a list of the notes registered at that month.

In [12]:
acled['date'] = pd.to_datetime(acled['event_date'])

# Create a 'month' column 
acled["month"] = acled["date"].dt.to_period("M").dt.to_timestamp()

# Group by country and month, then apply the aggregations
instructions_agg = {
    'fatalities': 'sum',
    'date': 'count',
    'notes': lambda x: " | ".join([str(item) for item in x if pd.notna(item) and str(item).lower() != 'nan'])
}

for evento in event_types:
    instructions_agg[evento] = 'sum'

acled = acled.groupby(['iso3', 'month']).agg(instructions_agg).reset_index()

acled = acled.rename(columns={
    'notes': 'notes_acled', 
    'date': 'event_count'
})

acled.head()

,iso3,month,fatalities,event_count,notes_acled,Battles,Explosions/Remote violence,Protests,Riots,Strategic developments,Violence against civilians
0,ABW,2018-03-01,0,1,"On 7 March 2018, in the morning, with the supp...",0,0,1,0,0,0
1,ABW,2018-04-01,0,1,"On 4 April 2018, in the morning, with the supp...",0,0,1,0,0,0
2,ABW,2018-09-01,0,1,"On 18 September 2018, in the morning, with the...",0,0,1,0,0,0
3,ABW,2019-02-01,0,1,"On 15 February 2019, in the morning, a group o...",0,0,1,0,0,0
4,ABW,2019-03-01,0,1,"On 29 March 2019, in the afternoon, around 75 ...",0,0,1,0,0,0


In [13]:
acled.to_csv("../data_clean/acled_clean.csv", index=False)

### HDX Signals

Regarding the data of HDX Signals, we have 1 file containing all the relevant information.

In [14]:
hdx = pd.read_csv("../data_raw/HDX Signals/hdx_signals.csv", sep=",")

In [15]:
hdx.head()

,iso3,location,region,hrp_location,indicator_id,date,alert_level,value,plot,map,...,summary_long,summary_short,summary_source,hdx_url,source_url,other_urls,further_information,campaign_url,campaign_date,signals_version
0,ARM,Armenia,Europe,False,wfp_market_monitor,2021-07-01 00:00:00,High concern,28.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,28% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfEA#ARM,2021-07-01,0.1.0
1,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2020-11-01 00:00:00,Medium concern,10.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,10% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfDc#BDI,2020-11-01,0.1.0
2,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2022-05-01 00:00:00,Medium concern,11.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,11% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfGA#BDI,2022-05-01,0.1.0
3,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2023-01-01 00:00:00,High concern,39.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,39% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfHM#BDI,2023-01-01,0.1.0
4,BEN,Benin,West and Central Africa,False,wfp_market_monitor,2020-11-01 00:00:00,Medium concern,21.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,21% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfDc#BEN,2020-11-01,0.1.0


First we want to check that all the observations have `iso3`and `date`, as both are imprecindible for the study. We will drop the observations without them.

In [16]:
hdx["date"] = pd.to_datetime(hdx["date"], errors="coerce")
hdx[hdx["date"].isna()]

,iso3,location,region,hrp_location,indicator_id,date,alert_level,value,plot,map,...,summary_long,summary_short,summary_source,hdx_url,source_url,other_urls,further_information,campaign_url,campaign_date,signals_version


In [17]:
hdx = hdx.dropna(subset=["date"]).copy()

In [18]:
hdx[hdx["iso3"].isna()]

,iso3,location,region,hrp_location,indicator_id,date,alert_level,value,plot,map,...,summary_long,summary_short,summary_source,hdx_url,source_url,other_urls,further_information,campaign_url,campaign_date,signals_version


In [19]:
hdx = hdx.dropna(subset=["iso3"]).copy()

Now we create the column `month` from `date`:

In [20]:
hdx["month"] = hdx["date"].dt.to_period("M").dt.to_timestamp()

In [21]:
hdx.head()

,iso3,location,region,hrp_location,indicator_id,date,alert_level,value,plot,map,...,summary_short,summary_source,hdx_url,source_url,other_urls,further_information,campaign_url,campaign_date,signals_version,month
0,ARM,Armenia,Europe,False,wfp_market_monitor,2021-07-01,High concern,28.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,28% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfEA#ARM,2021-07-01,0.1.0,2021-07-01
1,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2020-11-01,Medium concern,10.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,10% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfDc#BDI,2020-11-01,0.1.0,2020-11-01
2,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2022-05-01,Medium concern,11.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,11% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfGA#BDI,2022-05-01,0.1.0,2022-05-01
3,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2023-01-01,High concern,39.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,39% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfHM#BDI,2023-01-01,0.1.0,2023-01-01
4,BEN,Benin,West and Central Africa,False,wfp_market_monitor,2020-11-01,Medium concern,21.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,21% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfDc#BEN,2020-11-01,0.1.0,2020-11-01


In [22]:
hdx = hdx.rename(columns={"alert_level": "hdx_alert_level"})
hdx = hdx.rename(columns={"value": "hdx_value"})

Finally we group all the data for `iso3`x `month`, and we keep only the relevant columns.

In [23]:
# Create the dummy variables for the alert levels
dummies = pd.get_dummies(hdx['hdx_alert_level'], dtype=int, prefix='hdx_alert')
hdx = pd.concat([hdx, dummies], axis=1)
alert_types = dummies.columns.tolist()

instructions_agg = {
    'hdx_value': 'sum',
}

# Add the dummy columns
for alert in alert_types:
    instructions_agg[alert] = 'sum' 

hdx_grouped = hdx.groupby(['iso3', 'month']).agg(instructions_agg).reset_index()

# 4. Guardamos el resultado sin listas molestas
hdx_grouped.to_csv("../data_clean/hdx_clean.csv", index=False)


### EconAI

In [72]:
risk_3 = pd.read_csv("../data_raw/EconAI/conflictforecast_ons_armedconf_03.csv", low_memory=False)
logfat_3 = pd.read_csv("../data_raw/EconAI/conflictforecast_int_lnbest_03.csv", low_memory=False)
risk_12 = pd.read_csv("../data_raw/EconAI/conflictforecast_ons_armedconf_12.csv", low_memory=False)
logfat_12 = pd.read_csv("../data_raw/EconAI/conflictforecast_int_lnbest_12.csv", low_memory=False)

Let's first preprocess the files containing the risk prediction of the next 3 and 12 months.

In [73]:
risk_3.head()

,isocode,period,ons_armedconf_03_target,ons_armedconf_03_text,ons_armedconf_03_hist,ons_armedconf_03_all,ons_armedconf_03_naive,fatalities_UCDP,armedconf,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,201001,NaN,0.269263,0.998556,0.983878,1.0,344.0,1,0.999995,...,0.031562,0.147948,0.010954,0.002365,0.039283,0.041238,0.022845,0.002615,0.013236,112530.0
1,AFG,201002,NaN,0.257331,0.998499,0.991798,1.0,536.0,1,0.999995,...,0.033501,0.146538,0.011173,0.002334,0.038820,0.038723,0.022851,0.002433,0.012932,91058.0
2,AFG,201003,NaN,0.270456,0.998045,0.995489,1.0,407.0,1,0.999995,...,0.032701,0.138953,0.011799,0.002804,0.041742,0.036860,0.023159,0.002396,0.013774,102355.0
3,AFG,201004,NaN,0.261449,0.999598,0.999792,1.0,503.0,1,0.999995,...,0.034657,0.134142,0.012519,0.003110,0.043693,0.035465,0.024217,0.002324,0.014460,67243.0
4,AFG,201005,NaN,0.276905,0.995243,0.998218,1.0,502.0,1,0.999996,...,0.036816,0.131677,0.012806,0.002888,0.042615,0.034363,0.024350,0.002201,0.014451,76618.0


In [74]:
risk_12.head()

,isocode,period,ons_armedconf_12_target,ons_armedconf_12_text,ons_armedconf_12_hist,ons_armedconf_12_all,ons_armedconf_12_naive,fatalities_UCDP,armedconf,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,201001,NaN,0.534497,0.979979,0.987250,1.0,344.0,1,0.999995,...,0.031486,0.145819,0.010593,0.002425,0.039774,0.041119,0.023396,0.002611,0.013429,112530.0
1,AFG,201002,NaN,0.528127,0.982736,0.988044,1.0,536.0,1,0.999995,...,0.033440,0.144435,0.010810,0.002392,0.039358,0.038659,0.023398,0.002422,0.013095,91058.0
2,AFG,201003,NaN,0.533111,0.984456,0.989177,1.0,407.0,1,0.999995,...,0.032664,0.136855,0.011416,0.002888,0.042325,0.036741,0.023718,0.002390,0.013967,102355.0
3,AFG,201004,NaN,0.528683,0.986487,0.992189,1.0,503.0,1,0.999995,...,0.034651,0.132079,0.012109,0.003186,0.044280,0.035352,0.024767,0.002314,0.014655,67243.0
4,AFG,201005,NaN,0.533341,0.988304,0.992714,1.0,502.0,1,0.999996,...,0.036827,0.129605,0.012372,0.002955,0.043213,0.034278,0.024853,0.002194,0.014622,76618.0


In [75]:
risk_3 = risk_3.rename(columns={"isocode": "iso3"})
risk_3 = risk_3.rename(columns={"ons_armedconf_03_all": "risk_3"})
risk_3 = risk_3.rename(columns={"period": "month"})

# Convert period to datetime 
risk_3["month"] = pd.to_datetime(risk_3["month"], format="%Y%m")

risk_3.head()

,iso3,month,ons_armedconf_03_target,ons_armedconf_03_text,ons_armedconf_03_hist,risk_3,ons_armedconf_03_naive,fatalities_UCDP,armedconf,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,2010-01-01,NaN,0.269263,0.998556,0.983878,1.0,344.0,1,0.999995,...,0.031562,0.147948,0.010954,0.002365,0.039283,0.041238,0.022845,0.002615,0.013236,112530.0
1,AFG,2010-02-01,NaN,0.257331,0.998499,0.991798,1.0,536.0,1,0.999995,...,0.033501,0.146538,0.011173,0.002334,0.038820,0.038723,0.022851,0.002433,0.012932,91058.0
2,AFG,2010-03-01,NaN,0.270456,0.998045,0.995489,1.0,407.0,1,0.999995,...,0.032701,0.138953,0.011799,0.002804,0.041742,0.036860,0.023159,0.002396,0.013774,102355.0
3,AFG,2010-04-01,NaN,0.261449,0.999598,0.999792,1.0,503.0,1,0.999995,...,0.034657,0.134142,0.012519,0.003110,0.043693,0.035465,0.024217,0.002324,0.014460,67243.0
4,AFG,2010-05-01,NaN,0.276905,0.995243,0.998218,1.0,502.0,1,0.999996,...,0.036816,0.131677,0.012806,0.002888,0.042615,0.034363,0.024350,0.002201,0.014451,76618.0


In [76]:
risk_12 = risk_12.rename(columns={"isocode": "iso3"})
risk_12 = risk_12.rename(columns={"ons_armedconf_12_all": "risk_12"})
risk_12 = risk_12.rename(columns={"period": "month"})

# Convert period to datetime 
risk_12["month"] = pd.to_datetime(risk_12["month"], format="%Y%m")

risk_12.head()

,iso3,month,ons_armedconf_12_target,ons_armedconf_12_text,ons_armedconf_12_hist,risk_12,ons_armedconf_12_naive,fatalities_UCDP,armedconf,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,2010-01-01,NaN,0.534497,0.979979,0.987250,1.0,344.0,1,0.999995,...,0.031486,0.145819,0.010593,0.002425,0.039774,0.041119,0.023396,0.002611,0.013429,112530.0
1,AFG,2010-02-01,NaN,0.528127,0.982736,0.988044,1.0,536.0,1,0.999995,...,0.033440,0.144435,0.010810,0.002392,0.039358,0.038659,0.023398,0.002422,0.013095,91058.0
2,AFG,2010-03-01,NaN,0.533111,0.984456,0.989177,1.0,407.0,1,0.999995,...,0.032664,0.136855,0.011416,0.002888,0.042325,0.036741,0.023718,0.002390,0.013967,102355.0
3,AFG,2010-04-01,NaN,0.528683,0.986487,0.992189,1.0,503.0,1,0.999995,...,0.034651,0.132079,0.012109,0.003186,0.044280,0.035352,0.024767,0.002314,0.014655,67243.0
4,AFG,2010-05-01,NaN,0.533341,0.988304,0.992714,1.0,502.0,1,0.999996,...,0.036827,0.129605,0.012372,0.002955,0.043213,0.034278,0.024853,0.002194,0.014622,76618.0


Keep only the rows that we want (`iso3`, `period` and risk columns):

In [77]:
# Sort
risk_3 = risk_3.sort_values(["iso3", "month"])
risk_12 = risk_12.sort_values(["iso3", "month"])

cols_keep_3 = [
    "iso3",
    "month",
    "risk_3"
]

cols_to_keep_12 = [
    "iso3",
    "month",
    "risk_12"
]

risk_3 = risk_3[cols_keep_3].copy()
risk_12 = risk_12[cols_to_keep_12].copy()

Delete the rows that have missing values.

In [78]:
risk_3 = risk_3.dropna().reset_index(drop=True)
risk_12 = risk_12.dropna().reset_index(drop=True)

Now check the files logfat, containing the prediction of the number of fatalities of a country in the next 3 and 12 months.

In [79]:
logfat_3.head()

,isocode,period,int_lnbest_03_target,int_lnbest_03_text,int_lnbest_03_hist,int_lnbest_03_all,int_lnbest_03_naive,fatalities_UCDP,lnbest,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,201001,7.277248,6.357100,6.830939,6.844159,7.120444,344.0,5.843544,0.999995,...,0.031562,0.147948,0.010954,0.002365,0.039283,0.041238,0.022845,0.002615,0.013236,112530.0
1,AFG,201002,7.253470,6.358596,6.857611,6.883800,7.127694,536.0,6.285998,0.999995,...,0.033501,0.146538,0.011173,0.002334,0.038820,0.038723,0.022851,0.002433,0.012932,91058.0
2,AFG,201003,7.604396,6.466978,6.855770,6.898818,7.160846,407.0,6.011267,0.999995,...,0.032701,0.138953,0.011799,0.002804,0.041742,0.036860,0.023159,0.002396,0.013774,102355.0
3,AFG,201004,7.720905,6.363206,6.891226,6.939510,7.277248,503.0,6.222576,0.999995,...,0.034657,0.134142,0.012519,0.003110,0.043693,0.035465,0.024217,0.002324,0.014460,67243.0
4,AFG,201005,7.876638,6.494570,6.917318,6.938101,7.253470,502.0,6.220590,0.999996,...,0.036816,0.131677,0.012806,0.002888,0.042615,0.034363,0.024350,0.002201,0.014451,76618.0


In [80]:
logfat_12.head()

,isocode,period,int_lnbest_12_target,int_lnbest_12_text,int_lnbest_12_hist,int_lnbest_12_all,int_lnbest_12_naive,fatalities_UCDP,lnbest,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,201001,8.900004,6.612788,8.104944,8.102876,8.786609,344.0,5.843544,0.999995,...,0.031486,0.145819,0.010593,0.002425,0.039774,0.041119,0.023396,0.002611,0.013429,112530.0
1,AFG,201002,8.888205,6.433330,8.170137,8.129857,8.818186,536.0,6.285998,0.999995,...,0.033440,0.144435,0.010810,0.002392,0.039358,0.038659,0.023398,0.002422,0.013095,91058.0
2,AFG,201003,8.893847,6.608980,8.138193,8.131346,8.811503,407.0,6.011267,0.999995,...,0.032664,0.136855,0.011416,0.002888,0.042325,0.036741,0.023718,0.002390,0.013967,102355.0
3,AFG,201004,8.900685,6.563129,8.167914,8.149257,8.820847,503.0,6.222576,0.999995,...,0.034651,0.132079,0.012109,0.003186,0.044280,0.035352,0.024767,0.002314,0.014655,67243.0
4,AFG,201005,8.931420,6.572965,8.156089,8.141677,8.773694,502.0,6.220590,0.999996,...,0.036827,0.129605,0.012372,0.002955,0.043213,0.034278,0.024853,0.002194,0.014622,76618.0


In [81]:
logfat_3 = logfat_3.rename(columns={"isocode": "iso3"})
logfat_3 = logfat_3.rename(columns={"int_lnbest_03_all": "logfat_risk_3"})
logfat_3 = logfat_3.rename(columns={"period": "month"})

# Convert period to datetime 
logfat_3["month"] = pd.to_datetime(logfat_3["month"], format="%Y%m")

logfat_12 = logfat_12.rename(columns={"isocode": "iso3"})
logfat_12 = logfat_12.rename(columns={"int_lnbest_12_all": "logfat_risk_12"})
logfat_12 = logfat_12.rename(columns={"period": "month"})

# Convert period to datetime 
logfat_12["month"] = pd.to_datetime(logfat_12["month"], format="%Y%m")

logfat_3 = logfat_3.sort_values(["iso3", "month"])
logfat_12 = logfat_12.sort_values(["iso3", "month"])

cols_keep_3 = [
    "iso3",
    "month",
    "logfat_risk_3"
]

cols_to_keep_12 = [
    "iso3",
    "month",
    "logfat_risk_12"
]

logfat_3 = logfat_3[cols_keep_3].copy()
logfat_12 = logfat_12[cols_to_keep_12].copy()

Delete the rows that have missing values

In [82]:
logfat_3 = logfat_3.dropna().reset_index(drop=True)
logfat_12 = logfat_12.dropna().reset_index(drop=True)

Save all of them as a file together. First of all we are going to check that the four dataframes contain information of the same countries and months:

In [83]:
def get_countries(df, nom_df):
    if 'iso3' in df.index.names:
        return set(df.index.get_level_values('iso3').unique())
    elif 'iso3' in df.columns:
        return set(df['iso3'].unique())
    else:
        print(f"Column 'iso3' not found in {nom_df}")
        return set()

# Get countries for each DataFrame
countries_r3 = get_countries(risk_3, "risk_3")
countries_r12 = get_countries(risk_12, "risk_12")
countries_l3 = get_countries(logfat_3, "logfat_3")
countries_l12 = get_countries(logfat_12, "logfat_12")

# Check if they are identical
equal = (countries_r3 == countries_r12 == countries_l3 == countries_l12)

print("="*60)
print(f"The four DataFrames have the same countries: {'Yes' if equal else 'No'}")
print("="*60)

# If not equal, show details
if not equal:
    # Common countries in all 4 DataFrames
    common = countries_r3.intersection(countries_r12, countries_l3, countries_l12)
    # All unique countries that appear at least once
    all_detected = countries_r3.union(countries_r12, countries_l3, countries_l12)
    
    print(f"Common countries in all 4 ({len(common)}): {sorted(list(common))}\n")
    
    print("Details:")
    
    # Differences with risk_12
    if countries_r3 != countries_r12:
        if countries_r12 - countries_r3: print(f"  • In 'risk_12' left over: {countries_r12 - countries_r3}")
        if countries_r3 - countries_r12: print(f"  • In 'risk_12' missing: {countries_r3 - countries_r12}")
        
    # Differences with logfat_3
    if countries_r3 != countries_l3:
        if countries_l3 - countries_r3: print(f"  • In 'logfat_3' left over: {countries_l3 - countries_r3}")
        if countries_r3 - countries_l3: print(f"  • In 'logfat_3' missing: {countries_r3 - countries_l3}")
        
    # Differences with logfat_12
    if countries_r3 != countries_l12:
        if countries_l12 - countries_r3: print(f"  • In 'logfat_12' left over: {countries_l12 - countries_r3}")
        if countries_r3 - countries_l12: print(f"  • In 'logfat_12' missing: {countries_r3 - countries_l12}")
else:
    print(f"Detected countries ({len(countries_r3)}): {sorted(list(countries_r3))}")

The four DataFrames have the same countries: Yes
Detected countries (182): ['AFG', 'AGO', 'ALB', 'ARE', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS', 'BIH', 'BLR', 'BLZ', 'BMU', 'BOL', 'BRA', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF', 'CAN', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'COD', 'COL', 'COM', 'CRI', 'CUB', 'CYP', 'CZE', 'DEU', 'DJI', 'DNK', 'DOM', 'DZA', 'ECU', 'EGY', 'ERI', 'ESP', 'EST', 'ETH', 'FIN', 'FJI', 'FRA', 'GAB', 'GBR', 'GEO', 'GHA', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRC', 'GRD', 'GTM', 'GUY', 'HKG', 'HND', 'HRV', 'HTI', 'HUN', 'IDN', 'IND', 'IRL', 'IRN', 'IRQ', 'ISL', 'ISR', 'ITA', 'JAM', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ', 'KHM', 'KOR', 'KWT', 'LAO', 'LBN', 'LBR', 'LBY', 'LKA', 'LSO', 'LTU', 'LUX', 'LVA', 'MAC', 'MAR', 'MDA', 'MDG', 'MDV', 'MEX', 'MKD', 'MLI', 'MLT', 'MMR', 'MNE', 'MNG', 'MOZ', 'MRT', 'MUS', 'MWI', 'MYS', 'NAM', 'NER', 'NGA', 'NIC', 'NLD', 'NOR', 'NPL', 'NZL', 'OMN', 'PAK', 'PAN', 'PER', 'PHL', 'PNG', 'POL', 'PRI', 'PRK', 

In [84]:
econAI = (
    risk_3
    .merge(risk_12, on=["iso3", "month"])
    .merge(logfat_3, on=["iso3", "month"])
    .merge(logfat_12, on=["iso3", "month"])
)

In [85]:
econAI.head()

,iso3,month,risk_3,risk_12,logfat_risk_3,logfat_risk_12
0,AFG,2010-01-01,0.983878,0.987250,6.844159,8.102876
1,AFG,2010-02-01,0.991798,0.988044,6.883800,8.129857
2,AFG,2010-03-01,0.995489,0.989177,6.898818,8.131346
3,AFG,2010-04-01,0.999792,0.992189,6.939510,8.149257
4,AFG,2010-05-01,0.998218,0.992714,6.938101,8.141677


In [86]:
econAI.to_csv("../data_clean/econAI_clean.csv", index=False)

### INFORM INDEX

We are working with two different indexs. The first one is the inform index, and the second one the severity index.

Regarding the inform index, first we are going to check the workflow id for all the files to get the ones that we want.

In [87]:
import requests
import pandas as pd

url_workflows = "https://drmkc.jrc.ec.europa.eu/inform-index/API/InformAPI/Workflows"

print("Connecting to INFORM server...")

try:
    response = requests.get(url_workflows)
    workflows_data = response.json()
    
    df = pd.DataFrame(workflows_data)
    col_mapping = {col.lower(): col for col in df.columns}
    
    # Choose columns based on common names (case-insensitive)
    id_col = col_mapping.get('id', df.columns[0])
    name_col = col_mapping.get('name', df.columns[1] if len(df.columns) > 1 else df.columns[0])
    status_col = col_mapping.get('status', None)
    
    columns_to_show = [id_col, name_col]
    if status_col:
        columns_to_show.append(status_col)
        
    # Order by ID descending
    df_net = df[columns_to_show].sort_values(by=id_col, ascending=False)
    
    print("\n==========================================================================")
    print(" FILES AVAILABLE IN INFORM SERVER:")
    print("==========================================================================")
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_colwidth', None)
    print(df_net.to_string(index=False))

except Exception as e:
    print(f"\n❌ Error connecting to INFORM server: {e}")

Connecting to INFORM server...

 FILES AVAILABLE IN INFORM SERVER:
 WorkflowId                                              Name
        514                           INFORM Risk 2026 - 2017
        513                           INFORM Risk 2026 - 2018
        512                           INFORM Risk 2026 - 2019
        511                           INFORM Risk 2026 - 2020
        510                           INFORM Risk 2026 - 2021
        509                           INFORM Risk 2026 - 2022
        508                           INFORM Risk 2026 - 2023
        507                           INFORM Risk 2026 - 2024
        506                           INFORM Risk 2026 - 2025
        505                                  INFORM Risk 2026
        503                              INFORM Risk Mid 2025
        502               INFORM Risk 2025 2nd edition - 2016
        501               INFORM Risk 2025 2nd edition - 2017
        500               INFORM Risk 2025 2nd edition - 2018
   

Let's now extract the desired data:

In [88]:
## Load panel INFORM 2016-2026
## Keep publication_year and reference_year

import requests
import pandas as pd
import numpy as np

# publication year -> workflow id
workflow_mapping = {
    2016: 258,
    2017: 261,
    2018: 360,
    2019: 370,
    2020: 386,
    2021: 419,
    2022: 433,
    2023: 453,
    2024: 469,
    2025: 482,
    2026: 505
}

# publication year -> reference year
reference_year_mapping = {
    2016: 2015,
    2017: 2016,
    2018: 2017,
    2019: 2018,
    2020: 2020,
    2021: 2021,
    2022: 2022,
    2023: 2023,
    2024: 2024,
    2025: 2024,
    2026: 2025
}

# Indicators to keep
selected_indicators = ["INFORM", "VU", "CC", "HA"]

all_years = []

for publication_year, workflow_id in workflow_mapping.items():

    print(f"Processing INFORM publication year {publication_year}...")

    url = (
        f"https://drmkc.jrc.ec.europa.eu/inform-index/API/"
        f"InformAPI/Countries/Scores/?WorkflowId={workflow_id}"
    )

    data = requests.get(url).json()

    temp_df = pd.DataFrame(data)

    # keep only selected indicators
    temp_df = temp_df[
        temp_df["IndicatorId"].isin(selected_indicators)
    ]

   # pivot
    temp_df = temp_df.pivot(
    index="Iso3",
    columns="IndicatorId",
    values="IndicatorScore"
    ).reset_index()

    # remove pivot column name
    temp_df.columns.name = None

    # temporal metadata 
    temp_df["publication_year"] = publication_year
    temp_df["reference_year"] = reference_year_mapping[publication_year]

    # rearrenge columns
    temp_df = temp_df[
        [
            "Iso3",
            "publication_year",
            "reference_year",
            "INFORM",
            "VU",
            "CC",
            "HA"
        ]
    ]

    all_years.append(temp_df)

# combine all years into a single DataFrame
inform_df = pd.concat(all_years, ignore_index=True)

print(inform_df.head())

print(inform_df.shape)

Processing INFORM publication year 2016...
Processing INFORM publication year 2017...
Processing INFORM publication year 2018...
Processing INFORM publication year 2019...
Processing INFORM publication year 2020...
Processing INFORM publication year 2021...
Processing INFORM publication year 2022...
Processing INFORM publication year 2023...
Processing INFORM publication year 2024...
Processing INFORM publication year 2025...
Processing INFORM publication year 2026...
  Iso3  publication_year  reference_year  INFORM   VU   CC   HA
0  AFG              2016            2015     7.9  7.1  8.0  8.6
1  AGO              2016            2015     4.2  4.5  7.0  2.3
2  ALB              2016            2015     2.8  1.5  4.8  3.0
3  ARE              2016            2015     2.0  1.1  2.1  3.3
4  ARG              2016            2015     2.4  1.5  3.7  2.4
(2101, 7)


Now we have to expand the dataset to have a value for each month and country. Because the data is published at the end of march of each year, we are going to put the value of the index published in the year x, at all the months from 05-x to 04-(x+1):

In [89]:
# Create an empty list to store the temporary dataframes for each month
monthly_dfs = []

# Generate the 12 temporal shifts (from May to April of the following year)

# Months from May (05) to December (12) keep the same publication year
for month in range(5, 13):
    temp_df = inform_df.copy()
    temp_df['month'] = temp_df['publication_year'].astype(str) + '-' + f"{month:02d}"
    monthly_dfs.append(temp_df)

# Months from January (01) to April (04) belong to the NEXT calendar year (publication_year + 1)
for month in range(1, 5):
    temp_df = inform_df.copy()
    temp_df['month'] = (temp_df['publication_year'] + 1).astype(str) + '-' + f"{month:02d}"
    monthly_dfs.append(temp_df)

# Concatenate all monthly blocks into a single DataFrame
inform_panel = pd.concat(monthly_dfs, ignore_index=True)

# Standardize the month column format and sort the panel
inform_panel['month'] = pd.to_datetime(inform_panel['month'])
inform_panel = inform_panel.sort_values(by=['Iso3', 'month']).reset_index(drop=True)

# Rearrange columns to keep it clean
inform_panel = inform_panel[['Iso3', 'month', 'INFORM', 'VU', 'CC', 'HA']]
inform_panel = inform_panel.rename(columns={"Iso3": "iso3"})
inform_panel.head(15)

,iso3,month,INFORM,VU,CC,HA
0,AFG,2016-05-01,7.9,7.1,8.0,8.6
1,AFG,2016-06-01,7.9,7.1,8.0,8.6
2,AFG,2016-07-01,7.9,7.1,8.0,8.6
3,AFG,2016-08-01,7.9,7.1,8.0,8.6
4,AFG,2016-09-01,7.9,7.1,8.0,8.6
5,AFG,2016-10-01,7.9,7.1,8.0,8.6
6,AFG,2016-11-01,7.9,7.1,8.0,8.6
7,AFG,2016-12-01,7.9,7.1,8.0,8.6
8,AFG,2017-01-01,7.9,7.1,8.0,8.6
9,AFG,2017-02-01,7.9,7.1,8.0,8.6


In [90]:
inform_panel.to_csv("../data_clean/inform_clean.csv", index=False)

Now, regarding the severity index, we have one file containing all the available data since 2025-04, and then 12 other files containing the data of each month of the last year.

In [91]:
severity = pd.read_csv("../data_raw/INFORM/raw_inform_data_20260601.csv", sep=",")

In [92]:
severity.head()

,iso3,country,regions,crisis_id,crisis_name,inform_severity_index,impact_crisis,people_condition,complexity,drivers,date,country_level
0,AFG,Afghanistan,Asia,AFG001,Complex crisis in Afghanistan,4.4,4.8,4.4,4.2,"Conflict, Violence, Displacement, Drought, Earthquake, Socio-political",2019-01-01,Yes
1,AFG,Afghanistan,Asia,AFG002,Conflict in Afghanistan,4.1,4.6,3.6,4.2,Conflict,2019-01-01,No
2,AFG,Afghanistan,Asia,AFG003,Drought in Afghanistan,3.9,3.1,4.1,4.0,Drought,2019-01-01,No
3,AFG,Afghanistan,Asia,AFG001,Complex crisis in Afghanistan,4.4,4.8,4.4,4.2,"Conflict, Violence, Displacement, Drought, Earthquake, Socio-political",2019-02-01,Yes
4,AFG,Afghanistan,Asia,AFG001,Complex crisis in Afghanistan,4.4,4.8,4.4,4.2,"Conflict, Violence, Displacement, Drought, Earthquake, Socio-political",2019-03-01,Yes


We want to aggregate data for `iso3` and `month`. We have to delete the data that doesen't have one of these two values, create the month variable and group by these variables only keeping the informative columns.

In [93]:
severity[severity["iso3"].isna()]

,iso3,country,regions,crisis_id,crisis_name,inform_severity_index,impact_crisis,people_condition,complexity,drivers,date,country_level


In [94]:
severity[severity["date"].isna()]

,iso3,country,regions,crisis_id,crisis_name,inform_severity_index,impact_crisis,people_condition,complexity,drivers,date,country_level


In [95]:
severity = severity.dropna(subset=["iso3", "date"]).copy()

In [96]:
severity['date'] = pd.to_datetime(severity['date'])
severity["month"] = pd.to_datetime(severity["date"]).dt.to_period("M").dt.to_timestamp()

In [97]:
severity["inform_severity_index"] = pd.to_numeric(severity["inform_severity_index"], errors='coerce').astype('Float64')

In [98]:
severity[severity["inform_severity_index"].isna()]

,iso3,country,regions,crisis_id,crisis_name,inform_severity_index,impact_crisis,people_condition,complexity,drivers,date,country_level,month
145,ARM,Armenia,Middle east,ARM002,Nagorno-Karabakh Conflict in Armenia,<NA>,3.2,\N,1.9,"Conflict, Displacement",2020-11-01,Yes,2020-11-01
146,ARM,Armenia,Middle east,ARM002,Nagorno-Karabakh Conflict in Armenia,<NA>,3.2,\N,1.9,"Conflict, Displacement",2020-12-01,Yes,2020-12-01
152,ARM,Armenia,Middle east,REG013,Nagorno-Karabakh Conflict,<NA>,2.7,\N,2.9,NaN,2021-05-01,No,2021-05-01
154,ARM,Armenia,Middle east,REG013,Nagorno-Karabakh Conflict,<NA>,2.7,\N,2.7,NaN,2021-06-01,No,2021-06-01
156,ARM,Armenia,Middle east,REG013,Nagorno-Karabakh Conflict,<NA>,2.7,\N,2.6,NaN,2021-07-01,No,2021-07-01
158,ARM,Armenia,Middle east,REG013,Nagorno-Karabakh Conflict,<NA>,2.7,\N,2.6,NaN,2021-08-01,No,2021-08-01
160,ARM,Armenia,Middle east,REG013,Nagorno-Karabakh Conflict,<NA>,2.7,\N,2.6,NaN,2021-09-01,No,2021-09-01
162,ARM,Armenia,Middle east,REG013,Nagorno-Karabakh Conflict,<NA>,2.7,\N,2.6,NaN,2021-10-01,No,2021-10-01
164,ARM,Armenia,Middle east,REG013,Nagorno-Karabakh Conflict,<NA>,2.7,\N,2.6,NaN,2021-11-01,No,2021-11-01
166,ARM,Armenia,Middle east,REG013,Nagorno-Karabakh Conflict,<NA>,2.7,\N,2.6,NaN,2021-12-01,No,2021-12-01


We can't keep the rows with no data in the index variables, so we are going to drop these observations.

In [99]:
severity = severity.dropna(subset=["inform_severity_index"]).copy()

In [100]:
severity = severity.groupby(['iso3', 'month']).agg(
    inform_severity_index = ('inform_severity_index', 'mean'),
).reset_index()

In [101]:
severity.head(20)

,iso3,month,inform_severity_index
0,AFG,2019-01-01,4.133333
1,AFG,2019-02-01,4.4
2,AFG,2019-03-01,3.25
3,AFG,2019-04-01,3.35
4,AFG,2019-05-01,4.5
5,AFG,2019-06-01,4.5
6,AFG,2019-07-01,4.5
7,AFG,2019-08-01,4.5
8,AFG,2019-09-01,4.5
9,AFG,2019-10-01,4.4


Let's now preprocess the 12 files of the last year:

In [102]:
may = pd.read_excel(
    "../data_raw/INFORM/202505_INFORM_Severity_-_May_2025 (1).xlsx", 
    sheet_name="INFORM Severity - all crises",
    header=1
)
may = may.drop([0, 1]).reset_index(drop=True) # The first two rows are not needed, so we drop them and reset the index

june = pd.read_excel(
    "../data_raw/INFORM/202506_INFORM_Severity_-_June_2025 (1).xlsx", 
    sheet_name="INFORM Severity - all crises",
    header=1
)
june = june.drop([0, 1]).reset_index(drop=True)

july = pd.read_excel(
    "../data_raw/INFORM/202507_INFORM_Severity_-_July_2025.xlsx", 
    sheet_name="INFORM Severity - all crises",
    header=1
)
july = july.drop([0, 1]).reset_index(drop=True)

august = pd.read_excel(
    "../data_raw/INFORM/202508_INFORM_Severity_-_August_2025 (1).xlsx", 
    sheet_name="INFORM Severity - all crises",
    header=1
)
august = august.drop([0, 1]).reset_index(drop=True)

september = pd.read_excel(
    "../data_raw/INFORM/202509_inform_severity_-_september_2025.xlsx", 
    sheet_name="INFORM Severity - all crises",      
    header=1
)
september = september.drop([0, 1]).reset_index(drop=True)

october = pd.read_excel(
    "../data_raw/INFORM/202511_INFORM_Severity_-_mid_month_update-_November_2025 (1).xlsx", 
    sheet_name="INFORM Severity - all crises",      
    header=1
)
october = october.drop([0, 1]).reset_index(drop=True)   

november = pd.read_excel(
    "../data_raw/INFORM/202511_inform_severity_-_late_november_2025_ (1).xlsx", 
    sheet_name="INFORM Severity - all crises",      
    header=1
)
november = november.drop([0, 1]).reset_index(drop=True)

december = pd.read_excel(
    "../data_raw/INFORM/202512_inform_severity_mid_december_2025.xlsx", 
    sheet_name="INFORM Severity - all crises",      
    header=1
)
december = december.drop([0, 1]).reset_index(drop=True)

january = pd.read_excel(
    "../data_raw/INFORM/202601_INFORM_Severity_-_January_2026.xlsx", 
    sheet_name="INFORM Severity - all crises",      
    header=1
)
january = january.drop([0, 1]).reset_index(drop=True)

february = pd.read_excel(
    "../data_raw/INFORM/202602_INFORM_Severity_-_February_2026 (1).xlsx", 
    sheet_name="INFORM Severity - all crises",      
    header=1
)
february = february.drop([0, 1]).reset_index(drop=True)

march = pd.read_excel(
    "../data_raw/INFORM/202603_INFORM_Severity_-_March_2026 (1).xlsx", 
    sheet_name="INFORM Severity - all crises",      
    header=1
)
march = march.drop([0, 1]).reset_index(drop=True)

april = pd.read_excel(
    "../data_raw/INFORM/202604-inform-severity-april-2026 (2).xlsx", 
    sheet_name="INFORM Severity - all crises",      
    header=1
)
april = april.drop([0, 1]).reset_index(drop=True)

In [103]:
may.head()

,CRISIS,CRISIS ID,COUNTRY,ISO3,DRIVERS,INFORM Severity Index,INFORM Severity category,INFORM Severity category.1,Trend (last 3 months),Reliability,...,Geographical,Human,Conditions of people affected,People in need,Concentration of conditions,Complexity of the crisis,Society and safety,Operating environment,Regions,Last updated
0,Complex crisis in Afghanistan,AFG001,Afghanistan,AFG,"Conflict/ Violence,Political/economic crisis,Floods,Drought/drier conditions",4.5,5,Very High,Increasing,High,...,4.9,4.6,4.5,5,4,4.2,3.8,4.5,Asia,2025-05-20
1,Drought in South-West Angola,AGO002,Angola,AGO,Drought/drier conditions,3.2,4,High,Increasing,Medium,...,2.1,3.6,3.85,3.7,4,2.3,3.3,1,Africa,2025-05-23
2,Complex crisis in Burundi,BDI001,Burundi,BDI,"Floods,Political/economic crisis",3.3,4,High,Stable,Very High,...,4,3.1,3.25,3.5,3,3.2,2.9,3.5,Africa,2025-05-21
3,Displacement from Eastern DRC,BDI004,Burundi,BDI,International Displacement,1.8,2,Low,-,High,...,1.6,2.4,1.2,1.4,1,2.5,2.9,2,Africa,2025-05-20
4,Conflict in northern region of Benin,BEN002,Benin,BEN,Conflict/ Violence,2.4,3,Medium,Increasing,Very High,...,1.8,2.6,2.5,2,3,2.3,2.5,2,Africa,2025-05-30


In [104]:
may.columns

Index(['CRISIS', 'CRISIS ID', 'COUNTRY', 'ISO3', 'DRIVERS',
       'INFORM Severity Index', 'INFORM Severity category',
       'INFORM Severity category.1', 'Trend (last 3 months)', 'Reliability',
       'Impact of the crisis', 'Geographical', 'Human',
       'Conditions of people affected', 'People in need',
       'Concentration of conditions', 'Complexity of the crisis',
       'Society and safety', 'Operating environment', 'Regions',
       'Last updated'],
      dtype='str')

In [105]:
print(len(may[may["ISO3"].isna()]))
print(len(june[june["ISO3"].isna()]))
print(len(july[july["ISO3"].isna()]))
print(len(august[august["ISO3"].isna()]))
print(len(september[september["ISO3"].isna()]))
print(len(october[october["ISO3"].isna()]))
print(len(november[november["ISO3"].isna()]))
print(len(december[december["ISO3"].isna()]))
print(len(january[january["ISO3"].isna()]))
print(len(february[february["ISO3"].isna()]))
print(len(march[march["ISO3"].isna()]))
print(len(april[april["ISO3"].isna()]))

0
0
0
0
0
0
0
0
0
0
0
0


In [106]:
may['INFORM Severity Index'] = pd.to_numeric(may['INFORM Severity Index'], errors='coerce')
june['INFORM Severity Index'] = pd.to_numeric(june['INFORM Severity Index'], errors='coerce')
july['INFORM Severity Index'] = pd.to_numeric(july['INFORM Severity Index'], errors='coerce')
august['INFORM Severity Index'] = pd.to_numeric(august['INFORM Severity Index'], errors='coerce')
september['INFORM Severity Index'] = pd.to_numeric(september['INFORM Severity Index'], errors='coerce')
october['INFORM Severity Index'] = pd.to_numeric(october['INFORM Severity Index'], errors='coerce')
november['INFORM Severity Index'] = pd.to_numeric(november['INFORM Severity Index'], errors='coerce')
december['INFORM Severity Index'] = pd.to_numeric(december['INFORM Severity Index'], errors='coerce')
january['INFORM Severity Index'] = pd.to_numeric(january['INFORM Severity Index'], errors='coerce')
february['INFORM Severity Index'] = pd.to_numeric(february['INFORM Severity Index'], errors='coerce')
march['INFORM Severity Index'] = pd.to_numeric(march['INFORM Severity Index'], errors='coerce')
april['INFORM Severity Index'] = pd.to_numeric(april['INFORM Severity Index'], errors='coerce')

In [107]:
print(len(may[may["INFORM Severity Index"].isna()]))
print(len(june[june["INFORM Severity Index"].isna()]))
print(len(july[july["INFORM Severity Index"].isna()]))
print(len(august[august["INFORM Severity Index"].isna()]))
print(len(september[september["INFORM Severity Index"].isna()]))
print(len(october[october["INFORM Severity Index"].isna()]))
print(len(november[november["INFORM Severity Index"].isna()]))
print(len(december[december["INFORM Severity Index"].isna()]))
print(len(january[january["INFORM Severity Index"].isna()]))
print(len(february[february["INFORM Severity Index"].isna()]))
print(len(march[march["INFORM Severity Index"].isna()]))
print(len(april[april["INFORM Severity Index"].isna()]))

3
2
3
3
3
3
3
3
3
3
2
0


In [108]:
may.dropna(subset=['INFORM Severity Index'], inplace=True)
june.dropna(subset=['INFORM Severity Index'], inplace=True)
july.dropna(subset=['INFORM Severity Index'], inplace=True)
august.dropna(subset=['INFORM Severity Index'], inplace=True)
september.dropna(subset=['INFORM Severity Index'], inplace=True)
october.dropna(subset=['INFORM Severity Index'], inplace=True)
november.dropna(subset=['INFORM Severity Index'], inplace=True)
december.dropna(subset=['INFORM Severity Index'], inplace=True)
january.dropna(subset=['INFORM Severity Index'], inplace=True)
february.dropna(subset=['INFORM Severity Index'], inplace=True)
march.dropna(subset=['INFORM Severity Index'], inplace=True)
april.dropna(subset=['INFORM Severity Index'], inplace=True)

Creating the `month` column:

In [109]:
may['month'] = pd.to_datetime("2025-05").to_period("M").to_timestamp()
june['month'] = pd.to_datetime("2025-06").to_period("M").to_timestamp()
july['month'] = pd.to_datetime("2025-07").to_period("M").to_timestamp()
august['month'] = pd.to_datetime("2025-08").to_period("M").to_timestamp()
september['month'] = pd.to_datetime("2025-09").to_period("M").to_timestamp()
october['month'] = pd.to_datetime("2025-10").to_period("M").to_timestamp()
november['month'] = pd.to_datetime("2025-11").to_period("M").to_timestamp()
december['month'] = pd.to_datetime("2025-12").to_period("M").to_timestamp()

january['month'] = pd.to_datetime("2026-01").to_period("M").to_timestamp()
february['month'] = pd.to_datetime("2026-02").to_period("M").to_timestamp()
march['month'] = pd.to_datetime("2026-03").to_period("M").to_timestamp()
april['month'] = pd.to_datetime("2026-04").to_period("M").to_timestamp()

In [110]:
may.head()

,CRISIS,CRISIS ID,COUNTRY,ISO3,DRIVERS,INFORM Severity Index,INFORM Severity category,INFORM Severity category.1,Trend (last 3 months),Reliability,...,Human,Conditions of people affected,People in need,Concentration of conditions,Complexity of the crisis,Society and safety,Operating environment,Regions,Last updated,month
0,Complex crisis in Afghanistan,AFG001,Afghanistan,AFG,"Conflict/ Violence,Political/economic crisis,Floods,Drought/drier conditions",4.5,5,Very High,Increasing,High,...,4.6,4.5,5,4,4.2,3.8,4.5,Asia,2025-05-20,2025-05-01
1,Drought in South-West Angola,AGO002,Angola,AGO,Drought/drier conditions,3.2,4,High,Increasing,Medium,...,3.6,3.85,3.7,4,2.3,3.3,1,Africa,2025-05-23,2025-05-01
2,Complex crisis in Burundi,BDI001,Burundi,BDI,"Floods,Political/economic crisis",3.3,4,High,Stable,Very High,...,3.1,3.25,3.5,3,3.2,2.9,3.5,Africa,2025-05-21,2025-05-01
3,Displacement from Eastern DRC,BDI004,Burundi,BDI,International Displacement,1.8,2,Low,-,High,...,2.4,1.2,1.4,1,2.5,2.9,2,Africa,2025-05-20,2025-05-01
4,Conflict in northern region of Benin,BEN002,Benin,BEN,Conflict/ Violence,2.4,3,Medium,Increasing,Very High,...,2.6,2.5,2,3,2.3,2.5,2,Africa,2025-05-30,2025-05-01


Change the name of ISO3 so that it matches the other files:

In [111]:
may = may.rename(columns={'ISO3': 'iso3'})
june = june.rename(columns={'ISO3': 'iso3'})
july = july.rename(columns={'ISO3': 'iso3'})
august = august.rename(columns={'ISO3': 'iso3'})
september = september.rename(columns={'ISO3': 'iso3'})
october = october.rename(columns={'ISO3': 'iso3'})
november = november.rename(columns={'ISO3': 'iso3'})
december = december.rename(columns={'ISO3': 'iso3'})
january = january.rename(columns={'ISO3': 'iso3'})
february = february.rename(columns={'ISO3': 'iso3'})
march = march.rename(columns={'ISO3': 'iso3'})
april = april.rename(columns={'ISO3': 'iso3'})

Now we joing by month and country:

In [112]:
may = may.groupby(['iso3', 'month']).agg(
    inform_severity_index = ('INFORM Severity Index', 'mean'),
).reset_index()
june = june.groupby(['iso3', 'month']).agg(
    inform_severity_index = ('INFORM Severity Index', 'mean'),
).reset_index()
july = july.groupby(['iso3', 'month']).agg(
    inform_severity_index = ('INFORM Severity Index', 'mean'),
).reset_index()
august = august.groupby(['iso3', 'month']).agg(
    inform_severity_index = ('INFORM Severity Index', 'mean'),
).reset_index()
september = september.groupby(['iso3', 'month']).agg(
    inform_severity_index = ('INFORM Severity Index', 'mean'),
).reset_index()
october = october.groupby(['iso3', 'month']).agg(
    inform_severity_index = ('INFORM Severity Index', 'mean'),
).reset_index()
november = november.groupby(['iso3', 'month']).agg(
    inform_severity_index = ('INFORM Severity Index', 'mean'),
).reset_index()
december = december.groupby(['iso3', 'month']).agg(
    inform_severity_index = ('INFORM Severity Index', 'mean'),
).reset_index()
january = january.groupby(['iso3', 'month']).agg(
    inform_severity_index = ('INFORM Severity Index', 'mean'),
).reset_index()
february = february.groupby(['iso3', 'month']).agg(
    inform_severity_index = ('INFORM Severity Index', 'mean'),
).reset_index()
march = march.groupby(['iso3', 'month']).agg( 
    inform_severity_index = ('INFORM Severity Index', 'mean'),
).reset_index()
april = april.groupby(['iso3', 'month']).agg(
    inform_severity_index = ('INFORM Severity Index', 'mean'),
).reset_index() 

In [113]:
may.head()

,iso3,month,inform_severity_index
0,AFG,2025-05-01,4.50
1,AGO,2025-05-01,3.20
2,BDI,2025-05-01,2.55
3,BEN,2025-05-01,2.40
4,BFA,2025-05-01,4.20


Finally we cancatenate all the files, and we save the final one.

In [114]:
files = [
    severity, may, june, july, august, september, october, november, december,
    january, february, march, april
]

severity_final = pd.concat(files, ignore_index=True)
severity_final.head()

,iso3,month,inform_severity_index
0,AFG,2019-01-01,4.133333
1,AFG,2019-02-01,4.4
2,AFG,2019-03-01,3.25
3,AFG,2019-04-01,3.35
4,AFG,2019-05-01,4.5


In [115]:
severity_final.to_csv("../data_clean/severity_clean.csv", index=False)